In [1]:
"""
Modern Text-to-SQL System
LLM: Groq
Framework: LangChain
Vector Store: Chroma
Embeddings: HuggingFace
Database: MySQL

Architecture:
User Question
      ↓
Semantic Example Retrieval (Chroma)
      ↓
Few-Shot Prompt Construction
      ↓
Groq LLM SQL Generation
      ↓
SQL Query Checker
      ↓
Database Execution
      ↓
Answer
"""

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq

from langchain_community.utilities import SQLDatabase
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_experimental.sql import SQLDatabaseChain

from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from urllib.parse import quote_plus


# ------------------------------------------------
# 1. LOAD ENV VARIABLES
# ------------------------------------------------

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")


# ------------------------------------------------
# 2. INITIALIZE LLM
# ------------------------------------------------

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024
)


# ------------------------------------------------
# 3. CONNECT TO MYSQL DATABASE
# ------------------------------------------------



db_user = "root"
db_password = "Whitedevil123@"
db_host = "localhost"
db_name = "atliq_tshirts"

encoded_password = quote_plus(db_password)

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{db_user}:{encoded_password}@{db_host}:3306/{db_name}",
    sample_rows_in_table_info=3
)
print("✅ Connected to database")


# ------------------------------------------------
# 4. FEW SHOT EXAMPLES
# ------------------------------------------------

few_shots = [

{
"Question": "How many t-shirts do we have left for Nike in XS size and white color?",
"SQLQuery":
"""
SELECT SUM(stock_quantity)
FROM t_shirts
WHERE brand='Nike'
AND color='White'
AND size='XS'
""",
"SQLResult": "Result of the SQL query",
"Answer": "Number of Nike XS white t-shirts"
},

{
"Question": "How much is the total price of the inventory for all S-size t-shirts?",
"SQLQuery":
"""
SELECT SUM(price * stock_quantity)
FROM t_shirts
WHERE size='S'
""",
"SQLResult": "Result",
"Answer": "Total inventory value"
},

{
"Question": "How many white color Levi's shirts do we have?",
"SQLQuery":
"""
SELECT SUM(stock_quantity)
FROM t_shirts
WHERE brand='Levi'
AND color='White'
""",
"SQLResult": "Result",
"Answer": "Total Levi white shirts"
},

{
"Question": "How much revenue will be generated if all Levi t-shirts are sold without discount?",
"SQLQuery":
"""
SELECT SUM(price * stock_quantity)
FROM t_shirts
WHERE brand='Levi'
""",
"SQLResult": "Result",
"Answer": "Total revenue"
},

{
"Question": "What is revenue after discount for Levi shirts?",
"SQLQuery":
"""
SELECT SUM(price * stock_quantity * ((100-COALESCE(pct_discount,0))/100))
FROM t_shirts
LEFT JOIN discounts
ON t_shirts.t_shirt_id = discounts.t_shirt_id
WHERE brand='Levi'
""",
"SQLResult": "Result",
"Answer": "Discounted revenue"
}

]


# ------------------------------------------------
# 5. CREATE EMBEDDINGS
# ------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ------------------------------------------------
# 6. VECTOR STORE (CHROMA)
# ------------------------------------------------

to_vectorize = [
    " ".join(str(v) for v in example.values())
    for example in few_shots
]

vectorstore = Chroma.from_texts(
    texts=to_vectorize,
    embedding=embeddings,
    metadatas=few_shots,
    persist_directory="vector_db"
)


# ------------------------------------------------
# 7. SEMANTIC EXAMPLE SELECTOR
# ------------------------------------------------

example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2
)


# ------------------------------------------------
# 8. PROMPT TEMPLATE
# ------------------------------------------------

mysql_prompt = """
You are a MySQL expert.

Given a question, generate a syntactically correct MySQL query.

Rules:
- Use only columns that exist
- Do not hallucinate columns
- Use SUM(stock_quantity) when counting inventory
- Use SUM(price * stock_quantity) when calculating inventory value
- Use LEFT JOIN for discounts
- DO NOT wrap SQL in markdown or ```sql blocks
- Return ONLY the SQL query

Format:

Question: user question
SQLQuery: SQL query
SQLResult: result of the query
Answer: final natural language answer
"""


example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult", "Answer"],
    template="""
Question: {Question}

SQLQuery: {SQLQuery}

SQLResult: {SQLResult}

Answer: {Answer}
"""
)


# ------------------------------------------------
# 9. FEW SHOT PROMPT
# ------------------------------------------------

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix="""
Question: {input}
SQLQuery:
""",
    input_variables=["input"]
)


# ------------------------------------------------
# 10. SQL DATABASE CHAIN
# ------------------------------------------------

db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    prompt=few_shot_prompt,
    use_query_checker=True,
    return_intermediate_steps=True
)

import re

def clean_sql_query(query: str) -> str:
    """
    Remove markdown SQL blocks produced by LLM
    """
    query = re.sub(r"```sql", "", query, flags=re.IGNORECASE)
    query = re.sub(r"```", "", query)
    return query.strip()

# ------------------------------------------------
# 11. CHAT LOOP
# ------------------------------------------------

print("\n🚀 Text-to-SQL Assistant Ready\n")

while True:

    question = input("Ask a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    try:

        # Generate SQL with LLM
        sql_response = db_chain.llm_chain.invoke({"input": question})

        raw_sql = sql_response["text"]

        cleaned_sql = clean_sql_query(raw_sql)

        print("\n📊 Generated SQL:")
        print(cleaned_sql)

        # Execute manually
        result = db.run(cleaned_sql)

        print("\n✅ SQL Result:", result)

    except Exception as e:

        print("\n❌ Error:", e)

c:\Anaconda\envs\retail_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Connected to database


C:\Users\spars\AppData\Local\Temp\ipykernel_5100\3884527319.py:163: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7352.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Text-to-SQL Assistant Ready


📊 Generated SQL:
SELECT SUM(price * stock_quantity) 
FROM t_shirts 
WHERE size='S'

✅ SQL Result: [(Decimal('17372'),)]
